# De Bruijn graphs

In [ ]:
"""
de Bruijn Graph
Your group will be demonstrating your understanding of graph algorithms by implementing a de Bruijn graph algorithm

De Bruijn graphs are a powerful tool for representing and analyzing sequence data, especially in the context of genome assembly. They offer several advantages over other methods:

Key Features
Representation of Overlaps: De Bruijn graphs efficiently represent overlaps between sequences by using nodes to represent k-mers (subsequences of length k) and edges to show connections between these k-mers.
Handling Repeats: They are well-suited for representing genomes with repeats, which can be challenging for other assembly methods.
Scalability: De Bruijn graphs can handle large datasets more efficiently than some other approaches, making them suitable for modern high-throughput sequencing data.
Algorithm Structure
The de Bruijn graph algorithm for genome assembly typically involves the following steps:

Graph Construction: Create nodes for all (k-1)-mers from the set of k-mers in the sequencing reads.
Edge Creation: Draw edges between nodes if their corresponding k-mers overlap by k-1 bases.
Path Finding: Find an Eulerian path through the graph, which represents a possible reconstruction of the original sequence.
Sequence Reconstruction: Follow the path to reconstruct the original genome sequence.
Applications Beyond Genome Assembly
While de Bruijn graphs are primarily known for their use in genome assembly, they have applications in other areas of computer science and mathematics:

String Processing: Solving problems related to string overlaps and cyclic strings.
Network Design: Used in certain network topology designs.
Cryptography: Applied in some cryptographic algorithms.
Computational Considerations
It's important to note that while de Bruijn graphs are powerful, they come with computational challenges:

Memory Usage: They can be memory-intensive for large genomes or complex repeat structures.
Error Handling: Practical implementations need to account for sequencing errors and biological variations.
Parameter Tuning: The choice of k-mer size can significantly affect the assembly quality and computational requirements.
"""

You will be implementing one of the primary assembly algorithms from short-read data that is used today. We will implement a simple form of the algorithm where we _assume perfect sequencing_. That is, everything is sequenced exactly once and there are no errors or variants in the sequencing. 

A graph is composed of **nodes** and **edges** and we will need to develop a data strcture to track edges between nodes in our graph. We have provided the basic class structure as well as descriptions of functions to `add_edge` and `remove_edge` from the graph. You will need to implement these functions in order to then build the de Bruijn graph. 

In our implementation below, we use a `defaultdict` data structure to hold a list of all edges in the graph where all "right" nodes connected to a "left" node are stored in a list for that node.

```
build_debruijn_graph:
define substring length k and input string
For each k-length substring of input:
  split k mer into left and right k-1 mer
  add k-1 mers as nodes with a directed edge from left k-1 mer to right k-1 mer
```

---
## Eulerian walk

To continue our implementation from last class, we will use our De Bruijn graph to output a valid sequence from the assembly. This is implemented as a recursive algorithm by considering all valid edges. You will notice that as you change $k$, we are able to better recapitulate our sequence depending on how repetitive it is. In a more complex implementation of a Eulerian walk there are heuristics and defined rules for determining the validity of traversing a specific edge in the graph to result in a full graph-traversal. One of these methods is to traverse the graph in a depth first manner to avoid sectioning off any part of the graph in the traversal. In our implementation we will ignore these for simplicity.

```
eulerian_walk:
Beginning at first_node as node

For node:
    follow a random valid edge from node
    remove edge
    recurse
```


In [ ]:
import gzip

def read_fastq(filename):
    """Read sequences from a FASTQ file.
    Args:
        filename (str): Path to FASTQ file.
    Returns:
        list: List of DNA sequence strings (quality scores ignored).
    Example:
        >>> reads = read_fastq("test_reads.fastq")
        >>> len(reads)
        10
    """

    #Using the first 6 reads of the yeast fasta as my "test fasta"
    # The format is
        # @read_#
        # sequence
        # +
        # IIIIIIII
    # The sequence is on every 4th line, starting from 2 (2, 6, 10 etc)
    # So line_count % 4 == 2 appends only the sequence portion

    opener = gzip.open if filename.endswith(".gz") else open
    # gzip didn't work well for me, so I unzipped it from my terminal

    sequences = []
    with opener(filename, "rt") as fasta:
        for line_count, line in enumerate(fasta, 1):
            if line_count % 4 == 2:
                sequences.append(line.strip().upper())
    return sequences


In [8]:
# Checking logic of read_fastq()
seqlength = read_fastq("test_reads.fastq")
len(seqlength) #Expected output: 6 sequences

6

In [ ]:
"""
draw an edge between two k-1 mers only if the two k-1 mers are taken from the same read. Ex: GCT & CTA
"""

# Logic: if two k-1mers have an overlap of either prefix or suffix, we draw an edge between them, and then we can fact-check that the expected overlap is present
        # In the docstring example, we have 3-mers, so the k-1mer is a 2-mer
        # So -TG and TG- merge, since we don't want TG+TG, so its now A(TG)G

        # We need to check if end of a is in start of b, and we don't need to check if start of b is in end of a, because Eulerian walks are "linear" and this function is directed from left to right

        # Also, the two nodes are the two kmers, and the function args left and right

        # Also we need to split our long reads into kmers, I believe build_graph_from_reads does this

        # If a kmer repeats, maybe we should store each repeat in a list, instead of having one with a count of how many repeats, because then we can append the edge and remove it easier


        # It seems that we call add_edge() and do some operations to remove that edge
            # For parallel edges, we only want to remove the single occurrence each time
        # So would that mean that we first generate an edge from add_edge to see if one is present, if true, then merge them and remove the edge?

        #create a loop to check suffix of node a ("left") and prefix of node b ("right") using a sliding window
            #left = <something>[:-1]
            #right = <something>[1:]
            #Would it be better to access negative indexing like going backwards?

            #nodes as a set, edges as a list


            # start left to right, traceback right to left, will the output be in reverse order?

            # Look more into generators
            # use for eulerian walk so it doesnt take up memory, so we arent storing them

In [24]:
"""De Bruijn Graph Genome Assembly Module.

This module provides classes and functions for constructing De Bruijn graphs
from sequencing reads and performing genome assembly using Eulerian path
traversal.
"""
from collections import defaultdict
# provides default value for any key that doesn't exist yet

import random, sys
sys.setrecursionlimit(10**6)

class DeBruijnGraph:
    """Main class for De Bruijn graphs and genome assembly.

    This class builds De Bruijn graphs from sequencing reads and performs
    genome assembly by finding Eulerian paths through connected components
    of the graph.

    Attributes:
        graph (defaultdict): Adjacency list representation of graph edges.
            Keys are (k-1)-mers, values are lists of adjacent (k-1)-mers.
        k (int): The k-mer size used for graph construction.

    Example:
        >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
        >>> dbg = DeBruijnGraph(reads, k=6)
        >>> contigs = dbg.assemble_contigs(seed=42)
        >>> len(contigs) > 0
        True
    """
    def __init__(self, reads, k):
        """Initialize De Bruijn graph from sequencing reads.
        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer size for graph construction.
        Example:
            >>> reads = ["ATGGCG", "GCGTGC", "TGCAAC"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> len(dbg.graph) > 0
            True
        """
        self.graph = defaultdict(list)
        self.k = k
        self.build_graph_from_reads(reads, k)

    def add_edge(self, left, right):
        """Add a directed edge to the graph.
        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> "TGG" in dbg.graph["ATG"]
            True

        pseudocode:
        Based on the example: we want to add to the attribute in init

        since build_graph is what loops through the edges, we only need to define what a left and right node would be
            self.graph left and append to right
            keep right for no outgoing nodes

            No return statement because we update attributes

        """

        if right not in self.graph[left]:  # skip duplicates: Membership check
            self.graph[left].append(right)  # add the directed edge from left to right
        self.graph[right]  # make sure dead-end nodes exist as keys, even with no outgoing edges

        # No return because we're updating the attributes in __init__



    def remove_edge(self, left, right):
        """Remove a directed edge from the graph.
        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.
        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> dbg.remove_edge("ATG", "TGG")
            >>> len(dbg.graph["ATG"])
            0

        pseudocode:
        Based on the example: we want to remove the attribute in init
        self.graph left and remove right
        No return statement because we update attributes
        """

        self.graph[left].remove(right)




    def build_graph_from_reads(self, reads, k):
        """Build De Bruijn graph from multiple sequencing reads.

        Extracts all k-mers from all reads and adds edges between
        consecutive (k-1)-mers within each k-mer.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer length for graph construction.

        Example:
            >>> reads = ["ATGGC", "TGGCA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.build_graph_from_reads(reads, 4)
            >>> "ATG" in dbg.graph
            True

            Pseudocode:
            for each read within our reads
                and for i in range the length of our reads, but minus k
                    make a sliding window for the suffix/prefix
                    # define kmer, left, and right
                    call add_edge(left, right)
                no return statement
        """

        for read in reads:
            for i in range(0, len(read) - k + 1):
                kmer = read[i:i+k] # get the kmer
                left = kmer[:-1] # specify left prefix
                right = kmer[1:] # specify right suffix
                self.add_edge(left, right) # call add_edge




    def eulerian_walk(self, node, graph, seed=None, tour =None):
        """Perform recursive Eulerian walk on a graph component.

        This is a recursive function that follows all edges from a node
        to traverse the graph, building a path in reverse order.

        Args:
            node (str): Current node to traverse from.
            graph (defaultdict): Graph or subgraph to traverse.
            seed (int, optional): Seed for random edge selection.
            tour (list, optional): Shared path list filled in by every recursive call. Leave as None when calling from outside.            

        Returns:
            list: List of (k-1)-mers traversed (in reverse order).

        Example:
            >>> reads = ["ATGGCG"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> graph_copy = defaultdict(list, dbg.graph)
            >>> tour = dbg.eulerian_walk("ATG", graph_copy, seed=42)
            >>> len(tour) > 0
            True


        pseudocode:
        initialize current node and next node
            # node is already an argument, so we don't need to redefine
            # next_node = []
        Initialize the tour

        Loop body #maybe while node still has outgoing edge?
        Find outgoing edges from current node
            # Do we call add_edge?
        select the next node to traverse
        remove that edge
        use recursion
            # should nodes be recorded before or after a recursion call? If we blindly follow an edge, we might remove it without getting the correct path
            # nodes should be recorded after recursion

        We only want eulerian walk to handle one walk becuase assemble_contigs later will determine if there's disconnected branches or multiple contigs, according to its docstring

        # what do we do with the seed since its defined as none as an argument?
        Does it have to be if seed is not none , random number generator?

        remember that the sequence is going to be in reverse order

        """

        if seed is not None:
            random.seed(seed) 

        if tour is None: # only the outside call creates the list
            tour = [] #empty list to collect nodes

        while graph[node]: #while there are still edges
            index = random.randrange(len(graph[node])) #pick random edge position
            next_node = graph[node].pop(index) #remove that edge and save where it goes
            self.eulerian_walk(next_node, graph, tour=tour)  # go deeper, adding to the same list
            
        tour.append(node) # record the node once all its edges are used up
        return tour

    def _split_tour(self, tour):
        """Split a forward-order tour wherever two neighbors are not a real edge.

        The Eulerian walk always returns one long list of nodes. Usually every
        node in that list is connected to the next one by a real edge. But if
        the graph has a fork, the walk can finish one branch and then jump
        straight to another branch, even though there's no edge between them.
        Gluing those together would create a fake sequence.

        This helper goes through the list two nodes at a time and checks the
        original graph: if there's a real edge, the contig keeps growing; if
        not, the current contig ends and a new one starts.

        Args:
            tour (list): (k-1)-mers in forward order, from a reversed walk.

        Returns:
            list: One or more contig sequences (DNA strings).
        """
        contigs = []
        piece = [tour[0]]  # start the first contig with the first node
        for previous, current in zip(tour, tour[1:]):  # look at each neighboring pair
            if current in self.graph[previous]:  # real edge in the original graph: keep going
                piece.append(current)
            else:  # jump between branches: close this contig, start a new one
                contigs.append(self.tour_to_sequence(piece))
                piece = [current]
        contigs.append(self.tour_to_sequence(piece))  # add the last contig
        return contigs

    def assemble_contigs(self, seed=None):
        """Assemble all contigs from the De Bruijn graph.

        Finds all connected components and generates an Eulerian path
        for each component, producing multiple assembled contigs.

        Args:
            seed (int, optional): Random seed for reproducible assembly.

        Returns:
            list: List of assembled contig sequences (DNA strings).

        Example:
            >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
            >>> dbg = DeBruijnGraph(reads, k=6)
            >>> contigs = dbg.assemble_contigs(seed=42)
            >>> all(isinstance(c, str) for c in contigs)
            True

            pseudocode:
            seed the code
            # This is where we flip the sequence for tour


            
        """
        # Not sure yet where I want to put these to figure out the starting node
        # establish the edges that go in and out of the nodes
        #in_degree = defaultdict(int)
        #in_degree[right] += 1 # calculate the edges going in
        if seed is not None:
            random.seed(seed)
        graph_copy = defaultdict(list, {node: list(neighbors) for node, neighbors in self.graph.items()}) #make a copy of every node and its list of neighbors so popping from it doesnt affect original
         # Count arrows coming into each node
        in_degree = defaultdict(int)
        for left_node in self.graph:
            for right_node in self.graph[left_node]:
                in_degree[right_node] += 1

        # True beginnings: more arrows out (list length) than in
        starts = [node for node in self.graph
                  if len(self.graph[node]) > in_degree[node]]

        contigs = []
        # Starts first, then every node to catch closed loops
        for node in starts + list(graph_copy):
            if graph_copy[node]:  # skip nodes with no unused edges
                tour = self.eulerian_walk(node, graph_copy)
                tour.reverse()  # walk returns path backwards
                contigs.extend(self._split_tour(tour))

        return contigs

    def tour_to_sequence(self, tour):
        """Convert a tour of (k-1)-mers into a DNA sequence.

        Args:
            tour (list): List of (k-1)-mer strings in order.

        Returns:
            str: Assembled DNA sequence.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> tour = ['ATG', 'TGG', 'GGC', 'GCG']
            >>> dbg.tour_to_sequence(tour)
            'ATGGCG'
        """
        if not tour: #check if tour is empty
            return "" 
        sequence = tour[0]  #start with the first (k-1)-mer
        for node in tour[1:]: #go through every node after the first
            sequence += node[-1]
        return sequence



    def get_assembly_stats(self, contigs):
        """Calculate assembly statistics for assembled contigs.

        Args:
            contigs (list): List of contig sequences.

        Returns:
            dict: Dictionary containing assembly statistics:
                - num_contigs: Total number of contigs
                - total_length: Total assembled sequence length
                - longest_contig: Length of longest contig
                - shortest_contig: Length of shortest contig
                - mean_length: Mean contig length
                - n50: N50 statistic

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA", "CCCCCCCCCC"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> stats = dbg.get_assembly_stats(contigs)
            >>> stats['num_contigs']
            3
        """
        lengths = sorted((len(c) for c in contigs), reverse=True)  # longest first
        total = sum(lengths)

        # N50: length of the contig that pushes the running total past half
        n50 = 0
        running_total = 0
        for length in lengths:
            running_total += length
            if running_total >= total / 2:
                n50 = length
                break

        return {
            "num_contigs": len(lengths),
            "total_length": total,
            "longest_contig": lengths[0] if lengths else 0,  # guard: empty assembly
            "shortest_contig": lengths[-1] if lengths else 0,
            "mean_length": total / len(lengths) if lengths else 0.0,
            "n50": n50,
        }

    def write_fasta(self, contigs, filename, line_width=80):
        """Write assembled contigs to a FASTA file.

        Args:
            contigs (list): List of contig sequences.
            filename (str): Output FASTA filename.
            line_width (int, optional): Bases per sequence line (default 80)            

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.write_fasta(contigs, "output.fasta")
        """
        with open(filename, "w") as out:
            for i, contig in enumerate(contigs, 1):  # number contigs from 1
                out.write(f">contig_{i} length={len(contig)}\n")  # header line
                for start in range(0, len(contig), line_width):  # wrap into 80 bp lines
                    out.write(contig[start:start + line_width] + "\n")


In [25]:
# Test 6: FASTA output, including a contig long enough to wrap
dbg = DeBruijnGraph([], k=4)
dbg.write_fasta(["ATGGCG", "A" * 200], "test_output.fasta")
print(open("test_output.fasta").read())

>contig_1 length=6
ATGGCG
>contig_2 length=200
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA



In [23]:
# Test 5: assembly stats
dbg = DeBruijnGraph([], k=4)
print(dbg.get_assembly_stats(["ATGGCG", "TTTAAA", "CCCCCCCCCC"]))
print(dbg.get_assembly_stats([]))  # empty assembly should not crash

{'num_contigs': 3, 'total_length': 22, 'longest_contig': 10, 'shortest_contig': 6, 'mean_length': 7.333333333333333, 'n50': 6}
{'num_contigs': 0, 'total_length': 0, 'longest_contig': 0, 'shortest_contig': 0, 'mean_length': 0.0, 'n50': 0}


In [19]:
# TEST CELL 3 time taken for 50 kb chain

import time
random.seed(1)
long_read = "".join(random.choice("ACGT") for _ in range(50000))
dbg = DeBruijnGraph([long_read], k=31)

start = time.time()
contigs = dbg.assemble_contigs(seed=42)
print(f"Walk time: {time.time() - start:.2f} seconds")
print("Matches original read:", contigs[0] == long_read)

Walk time: 0.06 seconds
Matches original read: True


In [20]:
# Test 4: split a tour that jumps from E back to C
dbg = DeBruijnGraph([], k=2)
for left, right in [("A", "B"), ("B", "C"), ("B", "D"), ("C", "E"), ("D", "E")]:
    dbg.add_edge(left, right)
print(dbg._split_tour(["A", "B", "D", "E", "C", "E"]))

['ABDE', 'CE']


In [21]:
dbg = DeBruijnGraph([], k=2)
dbg.add_edge("A", "B")
dbg.add_edge("B", "C")
dbg.add_edge("B", "D")
dbg.add_edge("C", "E")
dbg.add_edge("D", "E")
contigs = dbg.assemble_contigs(seed=42)
print("Branching graph contigs:", contigs)

toy_reads = ["ATGGCGTACG", "GGCGTACGTT", "CGTACGTTAC", "TACGTTACCA", "CGTTACCATG",
             "TTACCATGGG", "ACCATGGGCC", "CATGGGCCTA", "TGGGCCTAAA"]
dbg = DeBruijnGraph(toy_reads, k=6)
print("Toy example contigs:", dbg.assemble_contigs(seed=42))
print("Expected:            ['ATGGCGTACGTTACCATGGGCCTAAA']")

Branching graph contigs: ['ABDE', 'CE']
Toy example contigs: ['ATGGCGTACGTTACCATGGGCCTAAA']
Expected:            ['ATGGCGTACGTTACCATGGGCCTAAA']


In [4]:
dbg = DeBruijnGraph(toy_reads, k=6)
for node in dbg.graph:
    dbg.graph[node] = list(dict.fromkeys(dbg.graph[node]))  # remove duplicate edges, keep order
print("Toy example, duplicates removed:", dbg.assemble_contigs(seed=42))

Toy example, duplicates removed: ['ATGGCGTACGTTACCATGGGCCTAAA']


---
# Toy Example

In [2]:
print("="*60)
print("EXAMPLE 1: Assembling 9 overlapping reads into one contig")
print("="*60)

toy_reads_1 = [
    "ATGGCGTACG",  # Read 1
    "GGCGTACGTT",  # Read 2: overlaps with Read 1
    "CGTACGTTAC",  # Read 3: overlaps with Read 2
    "TACGTTACCA",  # Read 4: overlaps with Read 3
    "CGTTACCATG",  # Read 5: overlaps with Read 4
    "TTACCATGGG",  # Read 6: overlaps with Read 5
    "ACCATGGGCC",  # Read 7: overlaps with Read 6
    "CATGGGCCTA",  # Read 8: overlaps with Read 7
    "TGGGCCTAAA"   # Read 9: overlaps with Read 8
]

print(f"\nInput: {len(toy_reads_1)} reads")
print("First read:  ", toy_reads_1[0])
print("Last read:   ", toy_reads_1[-1])
print(f"\nBuilding De Bruijn graph with k=6...")

EXAMPLE 1: Assembling 9 overlapping reads into one contig

Input: 9 reads
First read:   ATGGCGTACG
Last read:    TGGGCCTAAA

Building De Bruijn graph with k=6...


---
# Real data
This section utilizes a FASTQ file with 2.4 million "perfect" 150 bp reads simulated
from the _S. cerevisiae_ (yeast) genome (`R64`) and tasks your program to ingest, assemble, and traverse
the genome with statistical output report.

In [ ]:
"""Driver program for genome assembly using De Bruijn graphs.

This cell runs the completed DeBruijnGraph class on a FASTQ file of perfect
(error-free, variant-free) single-end reads simulated from a reference genome,
then reports assembly statistics.

Usage:
    Run all cells above this one first, then set the configuration values
    below and run this cell.

Expected input:
    - FASTQ or gzipped FASTQ (.fq, .fastq, .fq.gz, .fastq.gz)
    - Perfect single-end reads simulated with wgsim

Output:
    - <output_fasta>: Assembled contigs
    - <stats_file>: Detailed statistics report
"""

import time
from datetime import datetime

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
INPUT_FILE = "data/sacCer_R64_150SE_30x_wgsim.fq"
OUTPUT_FASTA = "genome_assembly.fasta"
STATS_FILE = "genome_assembly_stats.txt"
K_MER_SIZE = 31
# Size (bp) of the reference the reads were simulated from; used for the
# coverage and assembly-fraction estimates. Set to None if unknown.
GENOME_SIZE = 12_071_326


def _pct(part, whole):
    """Return part/whole as a percentage, or 0.0 when whole is zero."""
    return (part / whole * 100) if whole else 0.0


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    genome_size,
    coverage_estimate,
    assembly_fraction,
):
    """Write comprehensive assembly statistics to a text file.

    Args:
        stats_file (str): Output filename.
        input_file (str): Input FASTQ filename.
        num_reads (int): Number of reads processed.
        avg_read_length (float): Average read length.
        k_mer_size (int): K-mer size used.
        num_nodes (int): Number of graph nodes.
        num_edges (int): Number of distinct graph edges.
        stats (dict): Assembly statistics.
        contig_lengths (list): Contig lengths sorted longest first.
        timing (dict): Timing information, including 'total_time'.
        genome_size (int or None): Reference genome size in bp.
        coverage_estimate (float or None): Estimated sequencing coverage.
        assembly_fraction (float or None): Assembly size as % of genome.
    """
    rule = "-" * 80 + "\n"
    banner = "=" * 80 + "\n"

    with open(stats_file, "w") as f:
        f.write(banner)
        f.write("GENOME ASSEMBLY STATISTICS\n")
        f.write(banner + "\n")

        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input file:    {input_file}\n")
        f.write(f"K-mer size:    {k_mer_size}\n\n")

        f.write(rule + "INPUT DATA\n" + rule)
        f.write(f"Number of reads:         {num_reads:,}\n")
        f.write(f"Average read length:     {avg_read_length:.1f} bp\n")
        f.write(f"Total sequencing data:   {num_reads * avg_read_length:,.0f} bp\n")
        if genome_size:
            f.write(f"Reference genome size:   {genome_size:,} bp\n")
            f.write(f"Estimated coverage:      {coverage_estimate:.1f}x\n")
        f.write(f"Read time:               {timing['read_time']:.2f} seconds\n\n")

        f.write(rule + "DE BRUIJN GRAPH CONSTRUCTION\n" + rule)
        f.write(f"Graph nodes:             {num_nodes:,}\n")
        f.write(f"Graph edges:             {num_edges:,}\n")
        avg_degree = num_edges / num_nodes if num_nodes else 0.0
        f.write(f"Average out-degree:      {avg_degree:.2f}\n")
        f.write(f"Construction time:       {timing['graph_time']:.2f} seconds\n\n")

        f.write(rule + "ASSEMBLY RESULTS\n" + rule)
        f.write(f"Number of contigs:       {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length:   {stats['total_length']:,} bp\n")
        if genome_size:
            f.write(f"Assembly vs. genome:     {assembly_fraction:.2f}%\n")
        f.write(f"Longest contig:          {stats['longest_contig']:,} bp\n")
        f.write(f"Shortest contig:         {stats['shortest_contig']:,} bp\n")
        f.write(f"Mean contig length:      {stats['mean_length']:,.1f} bp\n")
        f.write(f"N50:                     {stats['n50']:,} bp\n")
        f.write(f"Assembly time:           {timing['assembly_time']:.2f} seconds\n\n")

        f.write(rule + "TOP 20 LONGEST CONTIGS\n" + rule)
        for i, length in enumerate(contig_lengths[:20], 1):
            f.write(f"{i:3d}. {length:10,} bp\n")
        f.write("\n")

        f.write(rule + "CONTIG LENGTH DISTRIBUTION\n" + rule)
        bins = [
            ("all", len(contig_lengths)),
            (">100kb", sum(1 for x in contig_lengths if x > 100_000)),
            (">50kb", sum(1 for x in contig_lengths if x > 50_000)),
            (">10kb", sum(1 for x in contig_lengths if x > 10_000)),
            (">5kb", sum(1 for x in contig_lengths if x > 5_000)),
            (">1kb", sum(1 for x in contig_lengths if x > 1_000)),
            (">500bp", sum(1 for x in contig_lengths if x > 500)),
        ]
        for bin_name, count in bins:
            f.write(f"Contigs {bin_name:8s}:     {count:,}\n")
        f.write("\n")

        total_time = timing["total_time"]
        f.write(rule + "TIMING SUMMARY (excluding file output)\n" + rule)
        for label, key in (
            ("Read time:", "read_time"),
            ("Graph construction:", "graph_time"),
            ("Assembly:", "assembly_time"),
        ):
            f.write(f"{label:25s}{timing[key]:8.2f} seconds "
                    f"({_pct(timing[key], total_time):5.1f}%)\n")
        f.write(f"{'Total time:':25s}{total_time:8.2f} seconds "
                f"({total_time / 60:.2f} minutes)\n\n")

        f.write(banner + "END OF REPORT\n" + banner)


def count_graph(graph):
    """Count nodes and distinct edges in an adjacency-list graph.

    Nodes that only receive edges may not be keys of the adjacency dict, so
    they are counted separately. Duplicate neighbor entries (the same k-mer
    seen in several reads) are counted once.

    Args:
        graph (dict): Mapping of (k-1)-mer -> iterable of neighbor (k-1)-mers.

    Returns:
        tuple: (num_nodes, num_edges)
    """
    num_edges = 0
    sink_only = set()
    for neighbors in graph.values():
        distinct = set(neighbors)
        num_edges += len(distinct)
        for node in distinct:
            if node not in graph:  # membership test does not insert keys
                sink_only.add(node)
    return len(graph) + len(sink_only), num_edges


def assemble_genome(
    input_file=INPUT_FILE,
    output_fasta=OUTPUT_FASTA,
    stats_file=STATS_FILE,
    k_mer_size=K_MER_SIZE,
    genome_size=GENOME_SIZE,
):
    """Run the complete assembly pipeline.

    1. Load FASTQ reads
    2. Build the De Bruijn graph
    3. Assemble contigs
    4. Calculate statistics
    5. Write output files

    Returns:
        dict: Assembly results with keys 'dbg', 'contigs', 'stats', 'timing',
            'graph_stats', 'coverage', 'output_fasta', and 'stats_file'.
    """
    print("=" * 80)
    print("GENOME ASSEMBLY PIPELINE")
    print("=" * 80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    timing = {}

    # ------------------------------------------------------------------ 1
    print("STEP 1: Loading sequencing reads")
    print("-" * 80)
    print(f"Input file: {input_file}\n")

    start_time = time.time()
    reads = read_fastq(input_file)
    timing["read_time"] = time.time() - start_time

    num_reads = len(reads)
    if num_reads == 0:
        raise ValueError(f"No reads found in {input_file}")
    total_bases = sum(len(read) for read in reads)
    avg_read_length = total_bases / num_reads

    print(f"Reads loaded: {num_reads:,}")
    print(f"Total bases: {total_bases:,} bp")
    print(f"Average read length: {avg_read_length:.1f} bp")
    print(f"Time elapsed: {timing['read_time']:.2f} seconds")
    print("\nSample reads:")
    print(f"  First: {reads[0][:80]}...")
    print(f"  Last:  {reads[-1][:80]}...\n")

    # ------------------------------------------------------------------ 2
    print("STEP 2: Building De Bruijn graph")
    print("-" * 80)
    print(f"K-mer size: {k_mer_size}")
    print(f"Building graph from {num_reads:,} reads...\n")

    start_time = time.time()
    dbg = DeBruijnGraph(reads, k=k_mer_size)
    timing["graph_time"] = time.time() - start_time

    del reads  # the graph holds everything needed; free the read list

    num_nodes, num_edges = count_graph(dbg.graph)
    avg_degree = num_edges / num_nodes if num_nodes else 0.0

    print("Graph construction complete!")
    print(f"  Nodes (distinct {k_mer_size - 1}-mers): {num_nodes:,}")
    print(f"  Edges (distinct {k_mer_size}-mers):   {num_edges:,}")
    print(f"  Average out-degree: {avg_degree:.2f}")
    print(f"Time elapsed: {timing['graph_time']:.2f} seconds\n")

    # ------------------------------------------------------------------ 3
    print("STEP 3: Assembling contigs")
    print("-" * 80)
    print("Traversing the graph to build contigs...\n")

    start_time = time.time()
    contigs = dbg.assemble_contigs()
    timing["assembly_time"] = time.time() - start_time
    timing["total_time"] = (
        timing["read_time"] + timing["graph_time"] + timing["assembly_time"]
    )

    print("Assembly complete!")
    print(f"  Contigs generated: {len(contigs):,}")
    print(f"Time elapsed: {timing['assembly_time']:.2f} seconds\n")

    # ------------------------------------------------------------------ 4
    print("STEP 4: Calculating assembly statistics")
    print("-" * 80)

    stats = dbg.get_assembly_stats(contigs)
    contig_lengths = sorted((len(c) for c in contigs), reverse=True)

    print("Assembly statistics:")
    print(f"  Number of contigs:     {stats['num_contigs']:,}")
    print(f"  Total assembly length: {stats['total_length']:,} bp")
    print(f"  Longest contig:        {stats['longest_contig']:,} bp")
    print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
    print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
    print(f"  N50:                   {stats['n50']:,} bp\n")

    print("Top 10 longest contigs:")
    for i, length in enumerate(contig_lengths[:10], 1):
        print(f"  {i:2d}. {length:,} bp")
    print()

    if genome_size:
        coverage_estimate = total_bases / genome_size
        assembly_fraction = _pct(stats["total_length"], genome_size)
        print("Genome coverage analysis:")
        print(f"  Reference genome size:         {genome_size:,} bp")
        print(f"  Estimated sequencing coverage: {coverage_estimate:.1f}x")
        print(f"  Assembly size vs. genome:      {assembly_fraction:.1f}%")
        print("  (Near 200% usually means reverse-complement duplicate "
              "contigs were not removed.)\n")
    else:
        coverage_estimate = assembly_fraction = None

    # ------------------------------------------------------------------ 5
    print("STEP 5: Writing output files")
    print("-" * 80)

    dbg.write_fasta(contigs, output_fasta)
    print(f"✓ Contigs written to: {output_fasta}")

    write_statistics_file(
        stats_file,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        genome_size,
        coverage_estimate,
        assembly_fraction,
    )
    print(f"✓ Statistics written to: {stats_file}\n")

    print("=" * 80)
    print("ASSEMBLY COMPLETE")
    print("=" * 80)
    print(f"Total time (excluding file output): {timing['total_time']:.2f} seconds "
          f"({timing['total_time'] / 60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    return {
        "dbg": dbg,
        "contigs": contigs,
        "stats": stats,
        "timing": timing,
        "graph_stats": {
            "num_nodes": num_nodes,
            "num_edges": num_edges,
            "avg_degree": avg_degree,
        },
        "coverage": coverage_estimate,
        "output_fasta": output_fasta,
        "stats_file": stats_file,
    }


print("\n" + "=" * 80)
print("GENOME DE BRUIJN GRAPH ASSEMBLY")
print("Student Assignment Driver Program")
print("=" * 80 + "\n")

result = assemble_genome()

print("=" * 80)
print("FIVE LONGEST ASSEMBLED CONTIGS")
print("=" * 80 + "\n")

for i, contig in enumerate(sorted(result["contigs"], key=len, reverse=True)[:5], 1):
    preview = contig[:100] + "..." if len(contig) > 100 else contig
    print(f">contig_{i} length={len(contig)}")
    print(preview + "\n")

print("=" * 80)
print("✓ ASSEMBLY COMPLETE")
print("=" * 80)
print("\nResults saved to:")
print(f"  • {result['output_fasta']} - {result['stats']['num_contigs']:,} assembled contigs")
print(f"  • {result['stats_file']} - detailed assembly statistics")
print("\nKey metrics:")
print(f"  • Total assembly: {result['stats']['total_length']:,} bp")
print(f"  • N50: {result['stats']['n50']:,} bp")
print(f"  • Longest contig: {result['stats']['longest_contig']:,} bp")
if result["coverage"] is not None:
    print(f"  • Coverage: {result['coverage']:.1f}x")
print()